In [44]:
import time, traceback, os, requests, json
import pandas as pd
from openpyxl import load_workbook
from sqlalchemy import inspect, text
from datetime import datetime
from zoneinfo import ZoneInfo

from db_utils import save_dataframe, get_engine
from api_utils import fetch_limtFull
from config import SIENGE_USERNAME, SIENGE_PASSWORD, POSTGRES_SCHEMA


In [45]:
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
db = os.getenv("POSTGRES_DB")
schema = os.getenv("POSTGRES_SCHEMA")
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")

_BU_units_


In [46]:
excel_file = "C:/Users/Moacir Faria/Olimpo Participacoes/OLP - CONTROLE/ANALISES/bancos_dados/Logs.xlsx"
sheet_name = "billId_documentNumber"
table_name = "tb_BU"

In [47]:
wb = load_workbook(excel_file, data_only=True)
ws = wb[sheet_name]

table = ws.tables[table_name]
ref = table.ref
cols_range = ref.split(":")
col_start = ''.join([c for c in cols_range[0] if c.isalpha()])
col_end   = ''.join([c for c in cols_range[1] if c.isalpha()])
usecols = f"{col_start}:{col_end}"
start_row = int(''.join([c for c in cols_range[0] if c.isdigit()]))

df = pd.read_excel(excel_file, sheet_name=sheet_name, usecols=usecols, header=0, skiprows=start_row-1)
df = df.dropna(subset=["bill_doc_number"])

In [50]:
engine = get_engine()
nome = table_name
df_on = df

In [43]:
with engine.begin() as conn:
    inspector = inspect(conn)
    if not inspector.has_table(nome):
        df_on.head(0).to_sql(nome, conn, if_exists='replace', index=False)
    conn.execute(text(f'TRUNCATE TABLE "{nome}" RESTART IDENTITY CASCADE'))
    df_on.to_sql(nome, conn, if_exists='append', index=False)


In [51]:
#print(len(df))
df_on[df_on["billId"]==10286]


,bill_doc_number,billId,documentIdentificationId,documentNumber,businessUnit,businessUnitInvoice,idDocument
9552,10286 / TREQ.12122025_2,10286.0,TREQ,12122025_2,Inter cia,Inter cia,Inter cia


_Baixar outcome_

In [3]:
url = "https://api.sienge.com.br/olimpo/public/api/bulk-data/v1/income?startDate=2014-01-01&endDate=2065-01-01&selectionType=D"
response = requests.get(url, auth=(api_user, api_password))

In [ ]:
if response.status_code == 200:
    dados = response.json()
    # caminho completo do arquivo
    caminho = r"C:\Users\Moacir Faria\Olimpo Participacoes\OLP - CONTROLE\BASES TRABALHADAS\JSON\outcome.json"
    # grava o JSON formatado
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False)                     #, indent=4)        # sem indent
    print("Arquivo salvo com sucesso em:", caminho)
else:
    print("Erro ao acessar API:", response.status_code, response.text)


In [24]:
dados = response.json()
df = pd.DataFrame(dados)
df_2 = pd.json_normalize(dados['data'])

In [22]:
base_IN = pd.json_normalize(df['data']).copy()
base_IN['creation_date'] = datetime.now(ZoneInfo("America/Sao_Paulo"))
base_IN.insert(0, 'id_base_in', range(1,  len(base_IN) + 1))
# Higienização
for c in ["documentIdentificationId", "mainUnit", "documentNumber"]:
    if c in base_IN.columns:
        base_IN[c] = base_IN[c].astype(str).str.strip()
base_IN = base_IN.rename(
    columns={
        "paymentTerm.id": "paymentTermid",
        "paymentTerm.descrition": "paymentTermdescrition"})

In [23]:
base_IN.head().T
#print(len(base_IN))

,0,1,2,3,4
id_base_in,1,2,3,4,5
companyId,20,20,20,20,20
companyName,LOTEAMENTO VILA BELLA PENAPOLIS SPE LTDA,LOTEAMENTO VILA BELLA PENAPOLIS SPE LTDA,LOTEAMENTO VILA BELLA PENAPOLIS SPE LTDA,LOTEAMENTO VILA BELLA PENAPOLIS SPE LTDA,LOTEAMENTO VILA BELLA PENAPOLIS SPE LTDA
businessAreaId,1,1,1,1,1
businessAreaName,AQUISIÇÃO DE CARTEIRA,AQUISIÇÃO DE CARTEIRA,AQUISIÇÃO DE CARTEIRA,AQUISIÇÃO DE CARTEIRA,AQUISIÇÃO DE CARTEIRA
projectId,NaN,NaN,NaN,NaN,NaN
projectName,None,None,None,None,None
groupCompanyId,1,1,1,1,1
groupCompanyName,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES
holdingId,None,None,None,None,None


In [34]:
#df_exp = df.explode("data").reset_index(drop=True)
#df_exp = df_exp.loc[df_exp["data"].apply(lambda x: isinstance(x, dict))]
# Normaliza o conteúdo de 'data' para colunas
base_OC = pd.json_normalize(df["data"]).copy()
base_OC["creation_date"] = datetime.now(ZoneInfo("America/Sao_Paulo"))
#base_OC.insert(0, "id_Base_oc", range(1, len(base_OC) + 1))
# Higienização
#for c in ["documentIdentificationId", "documentNumber"]:
#    if c in base_OC.columns:
#        base_OC[c] = base_OC[c].astype(str).str.strip()


In [35]:
base_OC.head()

,companyId,companyName,businessAreaId,businessAreaName,projectId,projectName,groupCompanyId,groupCompanyName,holdingId,holdingName,...,billDate,registeredUserId,registeredBy,registeredDate,paymentsCategories,departamentsCosts,buildingsCosts,payments,authorizations,creation_date
0,1,OLIMPO PARTICIPAÇÕES E EMPREENDIMENTOS IMOBILI...,1,ADMINISTRATIVO,NaN,None,2,GRUPO II - EMPRESAS GRUPO OLIMPO,None,None,...,2019-07-30,MAYARA,Mayara Lima,2019-09-01T17:36:49.305-03:00,"[{'costCenterId': 99915, 'costCenterName': 'OP...",[],[],[],"[{'authorizationUserId': 'MAYARA', 'authorizat...",2025-12-05 12:58:09.775412-03:00
1,15,OPMMR 02 EMPREENDIMENTOS IMOBILIARIOS SPE LTDA,2,INCORPORAÇÃO,NaN,None,1,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,None,None,...,2019-07-30,ALINEGNG,ALINE OLIVEIRA DE CARVALHO,2019-09-09T10:02:31.705-03:00,"[{'costCenterId': 1502, 'costCenterName': 'OBR...",[],"[{'buildingId': 15, 'buildingName': 'OPMMR 02 ...",[],"[{'authorizationUserId': 'SIDNEY', 'authorizat...",2025-12-05 12:58:09.775412-03:00
2,15,OPMMR 02 EMPREENDIMENTOS IMOBILIARIOS SPE LTDA,2,INCORPORAÇÃO,NaN,None,1,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,None,None,...,2019-09-04,ALINEGNG,ALINE OLIVEIRA DE CARVALHO,2019-09-19T09:40:37.001-03:00,"[{'costCenterId': 1502, 'costCenterName': 'OBR...",[],[],[],"[{'authorizationUserId': 'NATHALLIA', 'authori...",2025-12-05 12:58:09.775412-03:00
3,15,OPMMR 02 EMPREENDIMENTOS IMOBILIARIOS SPE LTDA,2,INCORPORAÇÃO,NaN,None,1,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,None,None,...,2019-09-05,ALINEGNG,ALINE OLIVEIRA DE CARVALHO,2019-09-19T13:41:17.873-03:00,"[{'costCenterId': 1502, 'costCenterName': 'OBR...",[],"[{'buildingId': 15, 'buildingName': 'OPMMR 02 ...",[],"[{'authorizationUserId': 'NATHALLIA', 'authori...",2025-12-05 12:58:09.775412-03:00
4,15,OPMMR 02 EMPREENDIMENTOS IMOBILIARIOS SPE LTDA,2,INCORPORAÇÃO,NaN,None,1,GRUPO I - SPE'S OLIMPO PARTICIPAÇÕES,None,None,...,2019-08-23,ALINEGNG,ALINE OLIVEIRA DE CARVALHO,2019-09-25T09:33:32.518-03:00,"[{'costCenterId': 1502, 'costCenterName': 'OBR...",[],"[{'buildingId': 15, 'buildingName': 'OPMMR 02 ...",[],"[{'authorizationUserId': 'SIDNEY', 'authorizat...",2025-12-05 12:58:09.775412-03:00


In [ ]:
#print(response.status_code)   # ex.: 200 significa sucesso
#print(response.reason)        # texto do status, ex.: "OK"
#len(response.content) / (1024*1024)


194456
Index(['data'], dtype='object')


_Baixar API sienge por request_


In [179]:
import os
import requests
import pandas as pd
from pandas import json_normalize
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()

True

In [180]:
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
db = os.getenv("POSTGRES_DB")
schema = os.getenv("POSTGRES_SCHEMA")
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")

_Customers_

In [181]:
all_data = []
offset = 0
limit = 200

while True:
    url = f"https://api.sienge.com.br/olimpo/public/api/v1/customers?limit={limit}&offset={offset}"
    response = requests.get(url, auth=(api_user, api_password))
    if response.status_code != 200:
        print("Erro:", response.status_code, response.text)
        break

    dados = response.json()
    results = dados.get("results", [])
    if not results:  # se não vier mais nada, encerra
        break
    all_data.extend(results)
    offset += limit
print(f"Total de registros baixados: {len(all_data)}")
df = pd.DataFrame(all_data)



Total de registros baixados: 3189


In [ ]:
# phones
df_ex = df.explode("phones").reset_index(drop=True)
df_ex = df_ex.drop_duplicates(subset=["id"]).reset_index(drop=True)
phones_df = json_normalize(df_ex["phones"]).add_prefix("phone_")
phones_df = phones_df.loc[df_ex.index].reset_index(drop=True)
df_1 = pd.concat([df_ex, phones_df], axis=1)

# addresses
df_ex = df_1.explode("addresses").reset_index(drop=True)
df_ex = df_ex.drop_duplicates(subset=["id"]).reset_index(drop=True)
addresses_df = json_normalize(df_ex["addresses"])
addresses_df = addresses_df.loc[df_ex.index].reset_index(drop=True)
df_2 = pd.concat([df_ex, addresses_df], axis=1)

# spouse
df_ex = df_2.copy()
spouse = df_ex["spouse"].apply(lambda x: x if isinstance(x, dict) else {})
spouse_df = json_normalize(spouse)[["cpf", "name", "email", "sex", "birthDate", "cellphoneNumber"]].add_prefix("spouse_")
df_3 = pd.concat([df_ex, spouse_df], axis=1)

df_3["familyIncome"] = df_3["familyIncome"].apply(lambda x: ",".join(map(str, x)) if isinstance(x, list) else x)
cols =["id","name","cpf","cnpj","numberIdentityCard","foreigner","personType","sex","nationality","birthDate","profession","civilStatus","matrimonialRegime","email",
       "phone_type","phone_idd","phone_number","phone_note","createdAt","mailingAddress","type","streetName","number","complement","neighborhood","city","state","zipCode",
       "spouse_name","spouse_cpf","spouse_sex","spouse_birthDate","spouse_email","spouse_cellphoneNumber","familyIncome"]
df_final = df_3.filter(items=cols)
print(df_final.to_string())


        id                                                                              name          cpf            cnpj numberIdentityCard foreigner personType        sex    nationality   birthDate                                     profession                civilStatus            matrimonialRegime                                          email   phone_type phone_idd     phone_number                     phone_note   createdAt mailingAddress           type                                                                                        streetName   number                                complement                              neighborhood                           city state    zipCode                                            spouse_name   spouse_cpf spouse_sex spouse_birthDate                                   spouse_email spouse_cellphoneNumber familyIncome
0     3435                                                          NEUSA GONÇALVES DOS REIS  33173555813             N

In [ ]:
#df_1.head() # imprime cabeçalho resumido com 5 linhas
#print(df_final.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df_final))
#print(df_1[df_1['phone_main'] != True])

        id                                                                              name          cpf            cnpj numberIdentityCard foreigner personType        sex    nationality   birthDate                                     profession                civilStatus            matrimonialRegime                                          email   phone_type phone_idd     phone_number                     phone_note   createdAt mailingAddress           type                                                                                        streetName   number                                complement                              neighborhood                           city state    zipCode                                            spouse_name   spouse_cpf spouse_sex spouse_birthDate                                   spouse_email spouse_cellphoneNumber familyIncome
0     3435                                                          NEUSA GONÇALVES DOS REIS  33173555813             N

_paymentCategory_

In [5]:
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")
url = "https://api.sienge.com.br/olimpo/public/api/v1/payment-categories"
response = requests.get(url, auth=(api_user, api_password))

In [6]:
if response.status_code == 200:
    dados = response.json()
    if dados:
        df = pd.DataFrame(dados)

        # Conexão com banco
        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")
        df.to_sql("financialCategories", engine, schema=schema, if_exists="replace", index=False)
        print("Dados importados com sucesso!")
    else:
        print("API respondeu, mas não retornou dados.")
else:
    print("Erro ao acessar API:", response.status_code, response.text)

Dados importados com sucesso!


In [7]:
# dicionário id->name para lookup
id_map = dict(zip(df["id"].astype(str), df["name"]))

def expand_row(row):
    codigo = str(row["id"])
    # pega prefixos progressivos que existam na base
    niveis = [codigo[:i] for i in range(1, len(codigo)+1) if codigo[:i] in id_map]
    cols = {}
    for j, n in enumerate(niveis[:-1]):  # exclui o último (que é o próprio R)
        cols[f"id{j+1}"] = n           # id correspondente
        cols[f"fc{j+1}"] = id_map[n]   # nome correspondente
    
    # adiciona colunas originais da linha
    for c in row.index:
        if c == "name":
            cols["financialCategory"] = row[c]
        else: cols[c] = row[c]
    return cols

# aplica apenas para tpConta = 'R'
df_out = pd.DataFrame([expand_row(r) for _, r in df[df["tpConta"]=="R"].iterrows()])

In [ ]:
def save_dataframe(df, table_name, schema):
#    engine = get_engine()
    df.to_sql(table_name, engine, schema=schema, if_exists="replace", index=False)
    print(f"Dados importados com sucesso na tabela {table_name}!")




In [ ]:
df_out.head()

,id1,fc1,id2,fc2,id3,fc3,id,financialCategory,tpConta,flRedutora,flAtiva,flAdiantamento,flImposto
0,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010101,Receita de Venda de Unidades Imobiliárias,R,N,S,N,N
1,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010102,Receita de Venda de Terreno,R,N,S,N,N
2,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010103,Receita de Frações Ideais,R,N,S,N,N
3,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010104,Receita de Unidades Vendidas em Permuta,R,N,S,N,N
4,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010105,Receita de Vendas de imóveis adq de Terceiros,R,N,S,N,N
